# Kiên Đoàn TTS — Nam trầm ấm

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [4]:
print('Đang cài đặt các thư viện cần thiết...')
!pip install -q --upgrade omnivoice gradio "numpy<2.1" "requests==2.32.4"
import site
from importlib import reload
reload(site)
print('Cài đặt hoàn tất!')

# Tải giọng mẫu
!wget -q https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples/nam-tram-am.mp3 -O voice_sample.mp3
print('Đã tải voice sample!')

Đang cài đặt các thư viện cần thiết...
Cài đặt hoàn tất!
Đã tải voice sample!


In [5]:
print('Đang trích xuất âm thanh từ video...')
!pip install -q moviepy
from moviepy.editor import VideoFileClip

try:
    video = VideoFileClip('/content/Download.mp4')
    video.audio.write_audiofile('extracted_voice.mp3')
    print('Đã trích xuất giọng từ video thành công!')
except Exception as e:
    print(f'Lỗi khi trích xuất: {e}')

Đang trích xuất âm thanh từ video...


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



MoviePy - Writing audio in extracted_voice.mp3


MoviePy - Done.
Đã trích xuất giọng từ video thành công!


In [6]:
# Cập nhật lại giá trị mặc định cho giao diện
def update_ui_with_extracted_voice():
    gr.close_all()
    with gr.Blocks(title='OmniVoice Timed SRT', theme=gr.themes.Soft()) as demo:
        gr.Markdown('# Kiên Đoàn TTS - Sử dụng giọng từ Video')
        with gr.Row():
            with gr.Column():
                ref = gr.Audio(label='1. Giọng mẫu (Đã lấy từ Video)', type='filepath', value='extracted_voice.mp3')
                srt = gr.File(label='2. Tải lên SRT', file_types=['.srt'])
                t = gr.Textbox(label='3. Nhập văn bản', lines=3)
                btn = gr.Button('Bắt đầu tạo', variant='primary')
            with gr.Column():
                out = gr.Audio(label='Kết quả (.wav)')
                final_text = gr.Textbox(label='Trạng thái', interactive=False)
        btn.click(generate_voice, inputs=[t, ref, srt], outputs=[out, final_text])
    demo.launch(share=True)

update_ui_with_extracted_voice()

  with gr.Blocks(title='OmniVoice Timed SRT', theme=gr.themes.Soft()) as demo:



Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae7c861d0b592164a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
print('Đang khởi động Omnivoice...')

import logging, os, re, time
import numpy as np
import torch
import gradio as gr
from datetime import datetime

# Patch: torch._utils removed in torch 2.13+
import torch as _torch
if not hasattr(_torch, '_utils'):
    _torch._utils = _torch._C._utils

import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Load model
DEVICE = get_best_device()
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate

# Configuration
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)

def srt_time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip().replace(',', '.'), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def parse_srt_with_timestamps(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    pattern = re.compile(r'(\d+)\n(\d{2}:\d{2}:\d{2},\d{3}) --> (\d{2}:\d{2}:\d{2},\d{3})\n(.*?)(?=\n\n|\Z)', re.DOTALL)
    matches = pattern.findall(content)

    subtitles = []
    for m in matches:
        start = srt_time_to_seconds(m[1])
        end = srt_time_to_seconds(m[2])
        text = m[3].replace('\n', ' ').strip()
        subtitles.append({'start': start, 'end': end, 'text': text})
    return subtitles

def generate_voice(text, ref_audio, srt_file):
    if ref_audio is None: return None, "Thiếu giọng mẫu."
    voice_prompt = model.create_voice_clone_prompt(ref_audio=ref_audio)

    if srt_file is not None:
        subs = parse_srt_with_timestamps(srt_file.name)
        full_audio = []
        current_time = 0.0

        for sub in subs:
            # Add silence gap
            silence_dur = sub['start'] - current_time
            if silence_dur > 0:
                full_audio.append(np.zeros(int(SAMPLING_RATE * silence_dur)))

            # Generate text
            a = model.generate(text=sub['text'], voice_clone_prompt=voice_prompt, language='vi', generation_config=GEN_CFG)[0]
            full_audio.append(a)
            current_time = sub['start'] + len(a) / SAMPLING_RATE

        audio = np.concatenate(full_audio)
        final_txt = "Đã xử lý từ file SRT với căn chỉnh thời gian."
    else:
        if not text.strip(): return None, "Nhập văn bản!"
        audio = model.generate(text=text, voice_clone_prompt=voice_prompt, language='vi', generation_config=GEN_CFG)[0]
        final_txt = text

    waveform = (audio * 32767).astype(np.int16)
    return (SAMPLING_RATE, waveform), final_txt

print('Khởi động giao diện...')
gr.close_all()
with gr.Blocks(title='OmniVoice Timed SRT', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Kiên Đoàn TTS - SRT Căn Chỉnh Thời Gian')
    with gr.Row():
        with gr.Column():
            ref = gr.Audio(label='1. Giọng mẫu', type='filepath', value='voice_sample.mp3')
            srt = gr.File(label='2. Tải lên SRT (Giữ đúng timeline)', file_types=['.srt'])
            t = gr.Textbox(label='3. Hoặc nhập văn bản', lines=3)
            btn = gr.Button('Bắt đầu tạo', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Kết quả (.wav)')
            final_text = gr.Textbox(label='Trạng thái', interactive=False)
    btn.click(generate_voice, inputs=[t, ref, srt], outputs=[out, final_text])

demo.launch(share=True)

Đang khởi động Omnivoice...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

...

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Khởi động giao diện...


/tmp/ipykernel_2627/1032501232.py:101: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='OmniVoice Timed SRT', theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c095213af06ec47f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
